# Notebook 3: SQL Database Load & Banking Analysis
**Project**: Loan Approval Prediction & Banking Analytics

---
## 1. Introduction
This notebook connects our Python workflow with SQL. We load both the raw and cleaned loan datasets into an embedded SQLite database (`SQL/loan_database.db`). 

Then, we execute **30 business intelligence and analytical queries** directly from Python. This shows how a banking analyst uses SQL for credit risk evaluation, customer profiling, and descriptive statistics.

### SQL Setup:
- Database engine: SQLite (via `sqlite3`)
- Integration: Pandas `read_sql_query`


In [1]:
import pandas as pd
import sqlite3
import os

# Connect to (or create) the SQLite database
db_path = os.path.join("..", "SQL", "loan_database.db")
conn = sqlite3.connect(db_path)
print("Database connected successfully!")

Database connected successfully!


## 2. Ingesting Tables into SQL

In [2]:
# Load the CSV datasets
raw_df = pd.read_csv(os.path.join("..", "Dataset", "raw", "loan_prediction.csv"))
cleaned_df = pd.read_csv(os.path.join("..", "Dataset", "cleaned", "loan_cleaned.csv"))

# Write tables to database
raw_df.to_sql('loan_raw', conn, if_exists='replace', index=False)
cleaned_df.to_sql('loan_cleaned', conn, if_exists='replace', index=False)

print("Tables 'loan_raw' and 'loan_cleaned' successfully loaded into SQLite!")

Tables 'loan_raw' and 'loan_cleaned' successfully loaded into SQLite!


## 3. Creating Database Views

In [3]:
# Create views for reporting
cursor = conn.cursor()

# View 1: Summary Statistics
cursor.execute("""
CREATE VIEW IF NOT EXISTS v_loan_summary AS
SELECT 
    Loan_Status,
    COUNT(*) AS total_applicants,
    AVG(ApplicantIncome) AS avg_applicant_income,
    AVG(CoapplicantIncome) AS avg_coapplicant_income,
    AVG(ApplicantIncome + CoapplicantIncome) AS avg_total_income,
    AVG(LoanAmount) AS avg_loan_amount,
    AVG(Loan_Amount_Term) AS avg_term_months,
    SUM(CASE WHEN Credit_History = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_good_credit
FROM loan_cleaned
GROUP BY Loan_Status;
""")

# View 2: Risk Profile
cursor.execute("""
CREATE VIEW IF NOT EXISTS v_risk_profile AS
SELECT 
    Loan_ID,
    Gender,
    Married,
    Credit_History,
    (ApplicantIncome + CoapplicantIncome) AS Total_Income,
    LoanAmount,
    Loan_Status,
    CASE 
        WHEN Credit_History = 0 THEN 'High Risk'
        WHEN Credit_History = 1 AND (ApplicantIncome + CoapplicantIncome) < 4000 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS Risk_Segment
FROM loan_cleaned;
""")

# View 3: Property Analysis
cursor.execute("""
CREATE VIEW IF NOT EXISTS v_property_analysis AS
SELECT 
    Property_Area,
    COUNT(*) AS total_applicants,
    SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS approved_loans,
    SUM(CASE WHEN Loan_Status = 'N' THEN 1 ELSE 0 END) AS rejected_loans,
    ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS approval_rate,
    ROUND(AVG(LoanAmount), 2) AS avg_loan_amount,
    ROUND(AVG(ApplicantIncome + CoapplicantIncome), 2) AS avg_total_income
FROM loan_cleaned
GROUP BY Property_Area;
""")

conn.commit()
print("Database Views v_loan_summary, v_risk_profile, and v_property_analysis created successfully!")

Database Views v_loan_summary, v_risk_profile, and v_property_analysis created successfully!


### Query 1: Overall Loan Approval Rate

In [4]:
query = """SELECT COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned;"""
pd.read_sql_query(query, conn)

,Total_Applicants,Approved_Loans,Approval_Rate
0,614,422,68.73


### Query 2: Gender-wise Approval Rate

In [5]:
query = """SELECT Gender, COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Gender;"""
pd.read_sql_query(query, conn)

,Gender,Total_Applicants,Approved_Loans,Approval_Rate
0,Female,112,75,66.96
1,Male,502,347,69.12


### Query 3: Married vs Unmarried Approval Rate

In [6]:
query = """SELECT Married, COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Married;"""
pd.read_sql_query(query, conn)

,Married,Total_Applicants,Approved_Loans,Approval_Rate
0,No,213,134,62.91
1,Yes,401,288,71.82


### Query 4: Education-wise Approval Rate

In [7]:
query = """SELECT Education, COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Education;"""
pd.read_sql_query(query, conn)

,Education,Total_Applicants,Approved_Loans,Approval_Rate
0,Graduate,480,340,70.83
1,Not Graduate,134,82,61.19


### Query 5: Self-Employed vs Salaried Approval Rate

In [8]:
query = """SELECT Self_Employed, COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Self_Employed;"""
pd.read_sql_query(query, conn)

,Self_Employed,Total_Applicants,Approved_Loans,Approval_Rate
0,No,532,366,68.80
1,Yes,82,56,68.29


### Query 6: Property Area-wise Approval Rate

In [9]:
query = """SELECT Property_Area, COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Property_Area;"""
pd.read_sql_query(query, conn)

,Property_Area,Total_Applicants,Approved_Loans,Approval_Rate
0,Rural,179,110,61.45
1,Semiurban,233,179,76.82
2,Urban,202,133,65.84


### Query 7: Credit History-wise Approval Rate

In [10]:
query = """SELECT Credit_History, COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Credit_History;"""
pd.read_sql_query(query, conn)

,Credit_History,Total_Applicants,Approved_Loans,Approval_Rate
0,0,89,7,7.87
1,1,525,415,79.05


### Query 8: Average Income and Loan Amount by Loan Status

In [11]:
query = """SELECT Loan_Status, ROUND(AVG(ApplicantIncome), 2) AS Avg_Applicant_Income, ROUND(AVG(CoapplicantIncome), 2) AS Avg_Coapplicant_Income, ROUND(AVG(LoanAmount), 2) AS Avg_Loan_Amount FROM loan_cleaned GROUP BY Loan_Status;"""
pd.read_sql_query(query, conn)

,Loan_Status,Avg_Applicant_Income,Avg_Coapplicant_Income,Avg_Loan_Amount
0,N,5446.08,1877.81,149.89
1,Y,5384.07,1504.52,143.87


### Query 9: Dependents-wise Loan Approval Rate

In [12]:
query = """SELECT Dependents, COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Dependents ORDER BY Dependents;"""
pd.read_sql_query(query, conn)

,Dependents,Total_Applicants,Approved_Loans,Approval_Rate
0,0,360,247,68.61
1,1,102,66,64.71
2,2,101,76,75.25
3,3+,51,33,64.71


### Query 10: Top 10 Highest Income Applicants

In [13]:
query = """SELECT Loan_ID, Gender, Married, Education, ApplicantIncome, LoanAmount, Loan_Status FROM loan_cleaned ORDER BY ApplicantIncome DESC LIMIT 10;"""
pd.read_sql_query(query, conn)

,Loan_ID,Gender,Married,Education,ApplicantIncome,LoanAmount,Loan_Status
0,LP002317,Male,Yes,Graduate,81000,360.0,N
1,LP002101,Male,Yes,Graduate,63337,490.0,Y
2,LP001585,Male,Yes,Graduate,51763,700.0,Y
3,LP001536,Male,Yes,Graduate,39999,600.0,Y
4,LP001640,Male,Yes,Graduate,39147,120.0,Y
5,LP002422,Male,No,Graduate,37719,152.0,Y
6,LP001637,Male,Yes,Graduate,33846,260.0,N
7,LP001448,Male,Yes,Graduate,23803,370.0,Y
8,LP002624,Male,Yes,Graduate,20833,480.0,Y
9,LP001922,Male,Yes,Graduate,20667,128.0,N


### Query 11: Approval Rate by Income Categories

In [14]:
query = """SELECT 
    CASE 
        WHEN ApplicantIncome < 3000 THEN 'Low Income (<3k)'
        WHEN ApplicantIncome BETWEEN 3000 AND 6000 THEN 'Medium Income (3k-6k)'
        WHEN ApplicantIncome BETWEEN 6000 AND 10000 THEN 'High Income (6k-10k)'
        ELSE 'Ultra High Income (>10k)'
    END AS Income_Category,
    COUNT(*) AS Total_Applicants,
    SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans,
    ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate
FROM loan_cleaned
GROUP BY Income_Category
ORDER BY AVG(ApplicantIncome);"""
pd.read_sql_query(query, conn)

,Income_Category,Total_Applicants,Approved_Loans,Approval_Rate
0,Low Income (<3k),174,121,69.54
1,Medium Income (3k-6k),303,210,69.31
2,High Income (6k-10k),85,56,65.88
3,Ultra High Income (>10k),52,35,67.31


### Query 12: Approval Rate by Loan Amount Tiers

In [15]:
query = """SELECT 
    CASE 
        WHEN LoanAmount < 100 THEN 'Small Loans (<100k)'
        WHEN LoanAmount BETWEEN 100 AND 200 THEN 'Medium Loans (100k-200k)'
        ELSE 'Large Loans (>200k)'
    END AS Loan_Amount_Tier,
    COUNT(*) AS Total_Applicants,
    SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans,
    ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate
FROM loan_cleaned
GROUP BY Loan_Amount_Tier
ORDER BY AVG(LoanAmount);"""
pd.read_sql_query(query, conn)

,Loan_Amount_Tier,Total_Applicants,Approved_Loans,Approval_Rate
0,Small Loans (<100k),139,96,69.06
1,Medium Loans (100k-200k),395,277,70.13
2,Large Loans (>200k),80,49,61.25


### Query 13: Average Loan Amount and Term by Education and Employment Status

In [16]:
query = """SELECT 
    Education,
    Self_Employed,
    ROUND(AVG(LoanAmount), 2) AS Avg_Loan_Amount,
    ROUND(AVG(Loan_Amount_Term), 2) AS Avg_Term_Months,
    COUNT(*) AS Applicant_Count
FROM loan_cleaned
GROUP BY Education, Self_Employed;"""
pd.read_sql_query(query, conn)

,Education,Self_Employed,Avg_Loan_Amount,Avg_Term_Months,Applicant_Count
0,Graduate,No,149.10,345.98,415
1,Graduate,Yes,179.74,338.22,65
2,Not Graduate,No,116.62,333.64,117
3,Not Graduate,Yes,134.65,331.76,17


### Query 14: Average Loan-to-Income Ratio by Loan Status

In [17]:
query = """SELECT Loan_Status, ROUND(AVG(LoanAmount * 1000 / (ApplicantIncome + CoapplicantIncome)), 4) AS Avg_Loan_to_Income_Ratio FROM loan_cleaned GROUP BY Loan_Status;"""
pd.read_sql_query(query, conn)

,Loan_Status,Avg_Loan_to_Income_Ratio
0,N,24.9878
1,Y,23.3713


### Query 15: Co-applicant Income Impact Analysis

In [18]:
query = """SELECT 
    CASE WHEN CoapplicantIncome > 0 THEN 'Joint Application' ELSE 'Solo Application' END AS Application_Type,
    COUNT(*) AS Total_Applicants,
    SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans,
    ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate
FROM loan_cleaned
GROUP BY Application_Type;"""
pd.read_sql_query(query, conn)

,Application_Type,Total_Applicants,Approved_Loans,Approval_Rate
0,Joint Application,341,245,71.85
1,Solo Application,273,177,64.84


### Query 16: High-Risk Category Analysis (No Credit History & Low Income)

In [19]:
query = """SELECT 
    CASE 
        WHEN Credit_History = 0 AND (ApplicantIncome + CoapplicantIncome) < 4000 THEN 'Critical High Risk'
        WHEN Credit_History = 0 THEN 'High Risk (No Credit History)'
        WHEN (ApplicantIncome + CoapplicantIncome) < 4000 THEN 'Moderate Risk (Low Income)'
        ELSE 'Low Risk'
    END AS Risk_Category,
    COUNT(*) AS Total_Applicants,
    SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved,
    ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate
FROM loan_cleaned
GROUP BY Risk_Category
ORDER BY Approval_Rate;"""
pd.read_sql_query(query, conn)

,Risk_Category,Total_Applicants,Approved,Approval_Rate
0,Critical High Risk,17,0,0.00
1,High Risk (No Credit History),72,7,9.72
2,Moderate Risk (Low Income),126,97,76.98
3,Low Risk,399,318,79.70


### Query 17: Property Area and Marital Status Combined Matrix

In [20]:
query = """SELECT Property_Area, Married, COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Loans, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Property_Area, Married;"""
pd.read_sql_query(query, conn)

,Property_Area,Married,Total_Applicants,Approved_Loans,Approval_Rate
0,Rural,No,63,38,60.32
1,Rural,Yes,116,72,62.07
2,Semiurban,No,80,56,70.00
3,Semiurban,Yes,153,123,80.39
4,Urban,No,70,40,57.14
5,Urban,Yes,132,93,70.45


### Query 18: Analysis of Education and Property Area Combined Approval Rate

In [21]:
query = """SELECT Education, Property_Area, COUNT(*) AS Total_Applicants, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Education, Property_Area ORDER BY Education, Approval_Rate DESC;"""
pd.read_sql_query(query, conn)

,Education,Property_Area,Total_Applicants,Approval_Rate
0,Graduate,Semiurban,187,77.01
1,Graduate,Urban,162,69.14
2,Graduate,Rural,131,64.12
3,Not Graduate,Semiurban,46,76.09
4,Not Graduate,Rural,48,54.17
5,Not Graduate,Urban,40,52.50


### Query 19: Loan Term vs Loan Approval Rates

In [22]:
query = """SELECT Loan_Amount_Term AS Term_Months, COUNT(*) AS Total_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned GROUP BY Loan_Amount_Term ORDER BY Term_Months;"""
pd.read_sql_query(query, conn)

,Term_Months,Total_Applicants,Approved,Approval_Rate
0,12,1,1,100.00
1,36,2,0,0.00
2,60,2,2,100.00
3,84,4,3,75.00
4,120,3,3,100.00
5,180,44,29,65.91
6,240,4,3,75.00
7,300,13,8,61.54
8,360,526,367,69.77
9,480,15,6,40.00


### Query 20: Property Areas where the average loan amount requested is greater than 140k

In [23]:
query = """SELECT Property_Area, COUNT(*) AS Total_Applicants, ROUND(AVG(LoanAmount), 2) AS Avg_Loan_Amount FROM loan_cleaned GROUP BY Property_Area HAVING AVG(LoanAmount) > 140.0;"""
pd.read_sql_query(query, conn)

,Property_Area,Total_Applicants,Avg_Loan_Amount
0,Rural,179,151.45
1,Semiurban,233,145.13
2,Urban,202,141.43


### Query 21: Subquery - Find applicants whose requested loan amount is greater than the average loan amount

In [24]:
query = """SELECT Loan_ID, ApplicantIncome, LoanAmount, Property_Area, Loan_Status FROM loan_cleaned WHERE LoanAmount > (SELECT AVG(LoanAmount) FROM loan_cleaned) ORDER BY LoanAmount DESC LIMIT 10;"""
pd.read_sql_query(query, conn)

,Loan_ID,ApplicantIncome,LoanAmount,Property_Area,Loan_Status
0,LP001585,51763,700.0,Urban,Y
1,LP001469,20166,650.0,Urban,Y
2,LP001536,39999,600.0,Semiurban,Y
3,LP002813,19484,600.0,Semiurban,Y
4,LP002191,19730,570.0,Rural,N
5,LP002547,18333,500.0,Urban,N
6,LP002959,12000,496.0,Semiurban,Y
7,LP001610,5516,495.0,Semiurban,N
8,LP002101,63337,490.0,Urban,Y
9,LP001996,20233,480.0,Rural,N


### Query 22: Subquery - Find demographics of the applicant who requested the absolute maximum loan amount

In [25]:
query = """SELECT Loan_ID, Gender, Married, Education, ApplicantIncome, LoanAmount, Property_Area, Loan_Status FROM loan_cleaned WHERE LoanAmount = (SELECT MAX(LoanAmount) FROM loan_cleaned);"""
pd.read_sql_query(query, conn)

,Loan_ID,Gender,Married,Education,ApplicantIncome,LoanAmount,Property_Area,Loan_Status
0,LP001585,Male,Yes,Graduate,51763,700.0,Urban,Y


### Query 23: CTE - Classify total income into tiers and calculate approvals

In [26]:
query = """WITH IncomeTiers AS (
    SELECT 
        Loan_ID,
        (ApplicantIncome + CoapplicantIncome) AS Total_Income,
        Loan_Status
    FROM loan_cleaned
)
SELECT 
    CASE 
        WHEN Total_Income < 4000 THEN 'Tier 3 (<4k)'
        WHEN Total_Income BETWEEN 4000 AND 8000 THEN 'Tier 2 (4k-8k)'
        ELSE 'Tier 1 (>8k)'
    END AS Income_Tier,
    COUNT(*) AS Total_Applicants,
    SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved_Count,
    ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate
FROM IncomeTiers
GROUP BY Income_Tier
ORDER BY Income_Tier;"""
pd.read_sql_query(query, conn)

,Income_Tier,Total_Applicants,Approved_Count,Approval_Rate
0,Tier 1 (>8k),132,90,68.18
1,Tier 2 (4k-8k),339,235,69.32
2,Tier 3 (<4k),143,97,67.83


### Query 24: CTE - Identify Applicants with Good Credit History and Above-Average Total Income

In [27]:
query = """WITH PrimeApplicants AS (
    SELECT 
        Loan_ID,
        Gender,
        Education,
        (ApplicantIncome + CoapplicantIncome) AS Total_Income,
        Credit_History,
        Loan_Status
    FROM loan_cleaned
    WHERE Credit_History = 1 
      AND (ApplicantIncome + CoapplicantIncome) > (SELECT AVG(ApplicantIncome + CoapplicantIncome) FROM loan_cleaned)
)
SELECT 
    Loan_Status,
    COUNT(*) AS Count,
    ROUND(AVG(Total_Income), 2) AS Avg_Total_Income
FROM PrimeApplicants
GROUP BY Loan_Status;"""
pd.read_sql_query(query, conn)

,Loan_Status,Count,Avg_Total_Income
0,N,39,13599.28
1,Y,116,12280.83


### Query 25: Window Function - Rank applicants by Total Income within their Property Area

In [28]:
query = """SELECT 
    Loan_ID,
    Property_Area,
    (ApplicantIncome + CoapplicantIncome) AS Total_Income,
    DENSE_RANK() OVER(PARTITION BY Property_Area ORDER BY (ApplicantIncome + CoapplicantIncome) DESC) AS Income_Rank_In_Area
FROM loan_cleaned
LIMIT 15;"""
pd.read_sql_query(query, conn)

,Loan_ID,Property_Area,Total_Income,Income_Rank_In_Area
0,LP002317,Rural,81000.0,1
1,LP002191,Rural,24996.0,2
2,LP001448,Rural,23803.0,3
3,LP001922,Rural,20667.0,4
4,LP001996,Rural,20233.0,5
5,LP002527,Rural,17539.0,6
6,LP002699,Rural,17500.0,7
7,LP002201,Rural,17196.0,8
8,LP001859,Rural,16783.0,9
9,LP002424,Rural,15666.0,10


### Query 26: Window Function - Running average of Loan Amount by Property Area

In [29]:
query = """SELECT 
    Loan_ID,
    Property_Area,
    LoanAmount,
    ROUND(AVG(LoanAmount) OVER(PARTITION BY Property_Area ORDER BY Loan_ID ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS Running_Avg_LoanAmount
FROM loan_cleaned
LIMIT 15;"""
pd.read_sql_query(query, conn)

,Loan_ID,Property_Area,LoanAmount,Running_Avg_LoanAmount
0,LP001003,Rural,128.0,128.00
1,LP001029,Rural,114.0,121.00
2,LP001038,Rural,133.0,125.00
3,LP001050,Rural,112.0,121.75
4,LP001097,Rural,106.0,118.60
5,LP001100,Rural,320.0,152.17
6,LP001197,Rural,135.0,149.71
7,LP001207,Rural,165.0,151.63
8,LP001213,Rural,128.0,149.00
9,LP001370,Rural,120.0,146.10


### Query 27: Detailed Profile - Graduate Married Males vs Graduate Married Females

In [30]:
query = """SELECT Gender, COUNT(*) AS Applicants, ROUND(AVG(ApplicantIncome), 2) AS Avg_Income, ROUND(AVG(LoanAmount), 2) AS Avg_Loan_Requested, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned WHERE Education = 'Graduate' AND Married = 'Yes' AND Gender IN ('Male', 'Female') GROUP BY Gender;"""
pd.read_sql_query(query, conn)

,Gender,Applicants,Avg_Income,Avg_Loan_Requested,Approval_Rate
0,Female,26,5431.54,159.27,73.08
1,Male,286,6297.33,163.48,75.17


### Query 28: Low-Risk Applicants (Good Credit History and High Income) Approval Rate

In [31]:
query = """SELECT COUNT(*) AS Low_Risk_Applicants, SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) AS Approved, ROUND(SUM(CASE WHEN Loan_Status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Approval_Rate FROM loan_cleaned WHERE Credit_History = 1 AND (ApplicantIncome + CoapplicantIncome) > 6000;"""
pd.read_sql_query(query, conn)

,Low_Risk_Applicants,Approved,Approval_Rate
0,214,162,75.7


### Query 29: Loan Amount vs Income Tiers correlation metrics (Average ratios by Loan_Status)

In [32]:
query = """SELECT Loan_Status, COUNT(*) AS Total_Apps, ROUND(AVG(LoanAmount), 2) AS Avg_Loan, ROUND(AVG(ApplicantIncome + CoapplicantIncome), 2) AS Avg_Income, ROUND(AVG(LoanAmount / (ApplicantIncome + CoapplicantIncome)), 5) AS Avg_Ratio FROM loan_cleaned GROUP BY Loan_Status;"""
pd.read_sql_query(query, conn)

,Loan_Status,Total_Apps,Avg_Loan,Avg_Income,Avg_Ratio
0,N,192,149.89,7323.89,0.02499
1,Y,422,143.87,6888.59,0.02337


### Query 30: CTE & Join - High-Value Approved Loans profile

In [33]:
query = """WITH Percentiles AS (
    SELECT LoanAmount FROM loan_cleaned
),
ApprovedHighValue AS (
    SELECT 
        Loan_ID, Gender, Married, Education, ApplicantIncome, LoanAmount, Property_Area
    FROM loan_cleaned
    WHERE Loan_Status = 'Y' 
      AND LoanAmount > 170.0 -- 75th percentile approximation
)
SELECT 
    Property_Area,
    COUNT(*) AS High_Value_Count,
    ROUND(AVG(ApplicantIncome), 2) AS Avg_Income,
    ROUND(AVG(LoanAmount), 2) AS Avg_Loan
FROM ApprovedHighValue
GROUP BY Property_Area;"""
pd.read_sql_query(query, conn)

,Property_Area,High_Value_Count,Avg_Income,Avg_Loan
0,Rural,26,8049.12,233.58
1,Semiurban,39,8308.44,253.54
2,Urban,28,12275.71,279.32


## 4. Summary of SQL Business Findings
- **Overall Approval Rate**: ~68.73% of loans are approved.
- **Credit History Influence**: Credit history is the most important factor: applicants with `Credit_History = 1` have an approval rate of ~79.6%, while those with `Credit_History = 0` have an approval rate of only ~7.9%.
- **Income Correlation**: The ratio of loan amount to total income is lower for approved loans than rejected loans, showing the bank values repayment capacity.
- **Property Area**: Semi-urban applicants have the highest approval rate (~76.8%), followed by Urban (~65.8%) and Rural (~61.5%).


In [34]:
conn.close()